In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import joblib
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

# 1. Configure MLflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("Greenhouse_Irrigation_Prediction")

# 2. Enable Autologging
mlflow.sklearn.autolog()


In [ ]:
df = pd.read_csv("../01_dataset/cropdata.csv")
df.head()


In [ ]:
df.shape


In [ ]:
df.dtypes


In [ ]:
obj_cols = df.select_dtypes(include=["object"]).columns
for col in obj_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())


In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].astype("category")
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [ ]:
df.dtypes


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[["MOI", "temp", "humidity"]] = scaler.fit_transform(
    df[["MOI", "temp", "humidity"]]
)


In [ ]:
import matplotlib.pyplot as plt

counts = df["result"].value_counts().sort_index()
plt.bar(counts.index.astype(str), counts.values, color="skyblue", edgecolor="black")
for i, v in enumerate(counts.values):
    plt.text(i, v + 0.2, str(v), ha="center", fontsize=10)

plt.xlabel("Result (0 = No Irrigation, 1 = Needs Irrigation)")
plt.ylabel("Count")
plt.title("Distribution of Irrigation Need")
plt.show()


In [ ]:
df = df[df["result"] != 2]


In [ ]:
X = df.drop(["result"], axis = 1)
y = df["result"]


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=10000, random_state=0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred)

# Note: mlflow.sklearn.autolog() handles parameter/metric/model logging automatically
print(f"Logistic Regression model accuracy: {acc:.2f}%")
print(f"F1 Score: {f1:.2f}")

# Explicitly save the model locally as requested
model_path = "../04_starter-kit/model.joblib"
joblib.dump(clf, model_path)
print(f"Model saved locally to: {model_path}")


In [ ]:
y_pred = clf.predict(X_test)


In [ ]:
f1 = f1_score(y_test, y_pred)
f1
